# EA1 – Carga y creación de dataset desde Hugging Face

**Analítica de Datos – IU Digital de Antioquia** · PREICA2602B020125 · Kevin Daniel Sepúlveda Giraldo

En este notebook coordino dos clases de Python que construí:

1. `CargadorHuggingFace` (`cargador_hf.py`): se conecta a Hugging Face y devuelve el dataset como DataFrame de pandas.
2. `ResumenDataFrame` (`resumen_dataframe.py`): recibe un DataFrame y guarda un resumen en un archivo `.txt`.

**Flujo:** clase de carga → DataFrame → transformaciones → clase de resumen → archivo `.txt`

**Dataset elegido:** [`aptologia/casa-vs-apartamento-precio-m2-ciudades-colombia-2026`](https://huggingface.co/datasets/aptologia/casa-vs-apartamento-precio-m2-ciudades-colombia-2026).
Compara el precio por metro cuadrado de casas y apartamentos en ciudades de Colombia (octubre de 2026).

## 1. Importar las clases

Las dos clases están en archivos `.py` dentro de la misma carpeta `EA1`, así que se importan directamente.

In [ ]:
import pandas as pd

from cargador_hf import CargadorHuggingFace
from resumen_dataframe import ResumenDataFrame

## 2. Obtener el DataFrame desde Hugging Face

Creo el objeto con el identificador del dataset y el split `train`.
Con `max_filas` dejo un límite para no traer más datos de los necesarios (recomendación de la clase para no romper el repositorio).

In [ ]:
DATASET = "aptologia/casa-vs-apartamento-precio-m2-ciudades-colombia-2026"

cargador = CargadorHuggingFace(DATASET, split="train", max_filas=1000)
df_original = cargador.cargar()

print(cargador.informacion())
df_original.head()

In [ ]:
df_original.info()

## 3. Transformaciones del DataFrame

Cada transformación devuelve un DataFrame nuevo, así conservo el original intacto.

### Transformación 1: renombrar columnas

Los nombres originales mezclan inglés y abreviaturas. Los cambio por nombres en español más claros.

In [ ]:
df_t1 = df_original.rename(columns={
    "city": "ciudad",
    "casa_cop_m2": "precio_casa_m2",
    "apto_cop_m2": "precio_apto_m2",
    "diferencia_pct": "diferencia_casa_vs_apto_pct",
})
df_t1.columns.tolist()

### Transformación 2: seleccionar columnas

Me quedo solo con las columnas que voy a analizar. Quito `n_si` y `n_no` porque
la ficha del dataset no explica qué significan, y no tiene sentido analizar algo que no puedo interpretar.

In [ ]:
df_t2 = df_t1[["ciudad", "precio_casa_m2", "precio_apto_m2", "diferencia_casa_vs_apto_pct"]]
print(f"Columnas antes: {df_t1.shape[1]} | Columnas después: {df_t2.shape[1]}")
df_t2.head()

### Transformación 3: crear columnas nuevas

- Paso los precios a **millones de pesos** (redondeados a 2 decimales) para que sean más fáciles de leer.
- Creo `tipo_mas_economico`, que indica si en esa ciudad sale más barato el m² de casa o de apartamento
  (si la diferencia es negativa, la casa es más barata).

In [ ]:
df_t3 = df_t2.assign(
    precio_casa_m2_millones=(df_t2["precio_casa_m2"] / 1_000_000).round(2),
    precio_apto_m2_millones=(df_t2["precio_apto_m2"] / 1_000_000).round(2),
    diferencia_casa_vs_apto_pct=df_t2["diferencia_casa_vs_apto_pct"].round(2),
    tipo_mas_economico=df_t2["diferencia_casa_vs_apto_pct"].apply(
        lambda dif: "Sin dato" if pd.isna(dif) else ("Casa" if dif < 0 else "Apartamento")
    ),
)
df_t3.head()

### Transformación 4: quitar columnas repetidas y ordenar

Como ya tengo los precios en millones, elimino las columnas en pesos y ordeno las ciudades
de mayor a menor diferencia a favor de la casa (las más negativas primero).

In [ ]:
df_transformado = (
    df_t3.drop(columns=["precio_casa_m2", "precio_apto_m2"])
         .sort_values("diferencia_casa_vs_apto_pct")
         .reset_index(drop=True)
)
df_transformado.head(10)

### Comparación antes y después

In [ ]:
print(f"Original:     {df_original.shape[0]} filas x {df_original.shape[1]} columnas")
print(f"Transformado: {df_transformado.shape[0]} filas x {df_transformado.shape[1]} columnas")
print("Columnas finales:", df_transformado.columns.tolist())

## 4. Generar el resumen en `.txt`

Le paso el DataFrame **transformado** a la segunda clase y guardo el archivo en la carpeta `resultados`.

In [ ]:
resumen = ResumenDataFrame(
    df_transformado,
    titulo="Resumen: precio m2 casa vs apartamento en ciudades de Colombia (2026)",
)
ruta_resumen = resumen.guardar("resultados/resumen_casa_vs_apartamento.txt")

print(f"Resumen guardado en: {ruta_resumen}\n")
print(ruta_resumen.read_text(encoding="utf-8"))

## 5. Resultado y explicación

**Transformaciones que apliqué:**

1. **Renombrar** las columnas a nombres claros en español.
2. **Seleccionar** solo las columnas útiles (se quitaron `n_si` y `n_no` porque no están documentadas).
3. **Crear columnas nuevas:** precios en millones de pesos y la columna `tipo_mas_economico`.
4. **Eliminar columnas repetidas y ordenar** las ciudades por la diferencia de precio.

El archivo `resultados/resumen_casa_vs_apartamento.txt` corresponde al DataFrame ya transformado.

In [ ]:
conteo = df_transformado["tipo_mas_economico"].value_counts()
print("Ciudades donde el m2 es más barato según el tipo de vivienda:")
print(conteo.to_string())

mayor = df_transformado.iloc[0]
print(f"\nMayor diferencia a favor de la casa: {mayor['ciudad']} "
      f"({mayor['diferencia_casa_vs_apto_pct']}%)")